# One-dimensional heat conduction with finite differences

Copyright © 2026 Philip Eisenlohr and contributors. [License](https://github.com/mseMSU/Notebooks-pub/blob/main/LICENSE.md)

In [ ]:
# Load the shared notebook utilities locally or, in Colab, from GitHub.
try:
    import nbkit
except ModuleNotFoundError:
    from pathlib import Path
    from urllib.request import urlretrieve
    import sys
    import tempfile

    module_path = next(
        (candidate
         for parent in (Path.cwd(), *Path.cwd().parents)
         for candidate in (parent / 'nbkit.py', parent / 'notebooks' / 'nbkit.py')
         if candidate.is_file()),
        None,
    )

    if module_path is None:
        support_dir = Path(tempfile.gettempdir()) / 'notebook_tools'
        support_dir.mkdir(exist_ok=True)
        module_path = support_dir / 'nbkit.py'
        urlretrieve(
            'https://raw.githubusercontent.com/'
            'mseMSU/Notebooks-pub/main/notebooks/nbkit.py',
            module_path,
        )

    sys.path.insert(0, str(module_path.parent))
    import nbkit


## Overview

This starter notebook illustrates how a finite-difference scheme turns a one-dimensional partial differential equation into repeated array updates.
We model a uniform rod and compare two ways its ends can interact with the surroundings: two fixed temperatures, followed by one fixed temperature and one insulated end.
The example is intentionally minimal and can be expanded into a broader lesson later.

After working through it, you should be able to identify a spatial grid, a time step, boundary conditions, the stability limit of an explicit scheme, and the role of a ghost point.

## Physical fields, partial differential equations, and discretization

Many physical problems concern a **field**, meaning a quantity that can vary with position and time.
Examples include temperature within a component, chemical concentration in a reactor, displacement within a solid, and fluid velocity or pressure in a pipe.
A field can be a scalar, such as temperature, or a vector, such as velocity.

The physical laws governing how fields evolve are commonly expressed as **partial differential equations**, or PDEs.
A PDE relates a field to its partial derivatives with respect to two or more independent variables, such as the spatial coordinates and time.
A general transport equation for a scalar field $u(\mathbf{x},t)$ can be written as

$$
\frac{\partial u}{\partial t}
+\mathbf{v}\cdot\nabla u
=D\nabla^2u+S.
$$

The time derivative describes local change, the term containing velocity $\mathbf{v}$ describes transport, the Laplacian $\nabla^2u$ describes spreading by diffusion, and $S$ represents sources or sinks.
Different physical assumptions can remove terms, add other terms, or couple several fields, producing a particular PDE for the problem at hand.

A continuous field has a value at infinitely many combinations of position and time, which a computer cannot store directly.
A numerical solution therefore begins with **discretization**: selecting a finite set of spatial locations and time levels at which approximate field values will be stored.
The spatial discretization determines where the field is represented, while the temporal discretization determines the sequence of times through which its evolution is calculated.

One simple choice, used throughout this notebook, is a fixed and uniformly spaced grid:

$$
x_i=i\,\Delta x, \qquad t_n=n\,\Delta t, \qquad \theta_i^n\approx\theta(x_i,t_n).
$$

The subscripts identify positions, the superscripts identify time levels, and $\Delta x$ and $\Delta t$ are the corresponding grid spacings.
The selected grid point and its four immediate neighbors illustrate the finite collection of stored values from which local changes can be estimated.

![A space–time grid with a central point and its four immediate neighbors labeled](assets/finite_difference_heat_equation/grid_neighbors.svg)

## Approximating derivatives on a fixed grid

The PDE is written in terms of derivatives of a continuous field, but the numerical representation contains only values at the selected grid points.
Finite differences approximate those derivatives using changes between nearby stored values.

### First derivatives

A difference between two neighboring samples measures the average slope between them, so a first-order difference naturally belongs halfway between the samples.

For example, the spatial and temporal forward differences are

$$
\left.\frac{\partial\theta}{\partial x}\right|_{i+1/2}^{n}\approx\frac{\theta_{i+1}^{n}-\theta_i^{n}}{\Delta x}, \qquad
\left.\frac{\partial\theta}{\partial t}\right|_{i}^{n+1/2}\approx\frac{\theta_i^{n+1}-\theta_i^n}{\Delta t}.
$$

![Forward spatial and temporal differences located between stored grid points](assets/finite_difference_heat_equation/first_derivatives.svg)

The half-indices in the diagram are conceptual locations rather than additional stored grid points.
They record where the difference quotient is naturally centered.
The spatial quotient between $i$ and $i+1$, for example, is associated with $i+1/2$ even though the array stores values only at integer indices.

If the first spatial derivative is needed at the stored point $i$, we can average the slopes on its left and right:

$$
\left.\frac{\partial\theta}{\partial x}\right|_i^n
\approx\frac{1}{2}\left(
\frac{\theta_i^n-\theta_{i-1}^n}{\Delta x}
+\frac{\theta_{i+1}^n-\theta_i^n}{\Delta x}
\right)
=\frac{\theta_{i+1}^n-\theta_{i-1}^n}{2\Delta x}.
$$

This is the centered-difference approximation to the first spatial derivative.
A centered temporal derivative at $t_n$ similarly uses values at $t_{n-1}$ and $t_{n+1}$.
During explicit time marching, however, $t_{n+1}$ is the unknown level we are trying to calculate, so we instead interpret the temporal quotient between $n$ and $n+1$ as a forward-Euler approximation at $t_n$.

### Second derivatives

A second spatial derivative measures how the first spatial derivative changes.
Define the two neighboring slope approximations as

$$
q_{i-1/2}^n=\frac{\theta_i^n-\theta_{i-1}^n}{\Delta x}, \qquad
q_{i+1/2}^n=\frac{\theta_{i+1}^n-\theta_i^n}{\Delta x}.
$$

These first derivatives occupy the two conceptual half-points on either side of $i$.
Their difference is centered back at the original stored grid point $i$.

![A second spatial derivative formed from neighboring first derivatives](assets/finite_difference_heat_equation/second_derivative.svg)

Dividing the change in these slopes by the distance between their half-point locations gives

$$
\left.\frac{\partial^2\theta}{\partial x^2}\right|_i^n
\approx\frac{1}{\Delta x}
\left(
\frac{\theta_{i+1}^n-\theta_i^n}{\Delta x}
-\frac{\theta_i^n-\theta_{i-1}^n}{\Delta x}
\right)
=\frac{\theta_{i+1}^n-2\theta_i^n+\theta_{i-1}^n}{\Delta x^2}.
$$

### Grid refinement and accuracy

Difference quotients approach the corresponding derivatives as $\Delta x$ and $\Delta t$ approach zero, provided the underlying solution is sufficiently smooth.
The centered first and second spatial differences above have truncation errors proportional to $\Delta x^2$, whereas the forward-Euler time approximation has an error proportional to $\Delta t$.
Consequently, halving $\Delta x$ ideally reduces the spatial discretization error by roughly a factor of four, while halving $\Delta t$ reduces the temporal discretization error by roughly a factor of two.
A finer grid also requires more stored values and more calculations.
For the explicit heat-equation scheme used below, reducing $\Delta x$ additionally forces a reduction of $\Delta t$ to preserve stability, so spatial refinement can substantially increase the number of time steps.
Comparing results on successively refined grids is therefore an important practical test that the numerical solution is converging.

## Toy model: one-dimensional heat conduction

Let $\theta(x,t)=T(x,t)-T_{\mathrm{ref}}$ denote temperature excess above a reference temperature.
For a uniform rod with constant thermal diffusivity $\alpha$ and no internal heat generation, conservation of energy leads to the heat equation

$$
\frac{\partial \theta}{\partial t}=\alpha\frac{\partial^2\theta}{\partial x^2}, \qquad 0<x<L.
$$

Here, $L$ is measured in metres, time in seconds, $\alpha$ in square metres per second, and $\theta$ in kelvin.
The time derivative describes local change, while the second spatial derivative measures curvature and therefore how different a point is from its surroundings.

## Initial and boundary conditions

The differential equation alone does not determine a unique temperature history.
We must specify the temperature throughout the rod at the initial time and describe how both ends interact with their surroundings.

Common boundary-condition choices include:

- A **fixed value** or Dirichlet condition prescribes the temperature at an end, such as $\theta(0,t)=0$.
- A **fixed slope** or Neumann condition prescribes $\partial\theta/\partial x$ and therefore the conductive heat flux; a zero slope represents an insulated end.
- A **mixed** or Robin condition relates temperature to its slope and can model convective heat transfer to the surroundings.
- A **periodic** condition connects the two ends and is useful when the modeled interval repeats.

Dirichlet conditions are especially simple numerically because their boundary-node values can be assigned directly after every time step.
Slope and mixed conditions require an additional finite-difference approximation at the boundary, often using a one-sided difference or an auxiliary ghost point.
An initial profile should also satisfy the selected boundary conditions so that the model does not begin with an incompatibility at a corner of the space–time domain.

## Assemble the explicit solution scheme

We approximate the time derivative between levels $n$ and $n+1$ with a forward difference and evaluate the centered second spatial derivative using values from level $n$:

$$
\frac{\theta_i^{n+1}-\theta_i^n}{\Delta t}
=\alpha\frac{\theta_{i+1}^n-2\theta_i^n+\theta_{i-1}^n}{\Delta x^2}.
$$

Solving this algebraic equation for the only unknown, $\theta_i^{n+1}$, gives the explicit update

$$
\theta_i^{n+1}=\theta_i^n+r\left(\theta_{i+1}^n-2\theta_i^n+\theta_{i-1}^n\right), \qquad r=\frac{\alpha\Delta t}{\Delta x^2}.
$$

Every interior value at the new time level must be calculated from the same old time level.
How the boundary nodes are updated depends on the physical boundary conditions and will be specified separately for each case below.
For this constant-coefficient, one-dimensional explicit scheme on a uniform grid, stability requires $r\leq 1/2$; see [Chapter 20, *Finite difference schemes for the heat equation in one dimension*](https://userpages.umbc.edu/~rostamia/cbook/fd1/finite-differences.pdf) for a derivation.
A time step that violates this restriction can make rounding and discretization errors grow until the numerical result oscillates or diverges.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

L = 0.1                         # Rod length, m
alpha = 1.0e-4                  # Thermal diffusivity, m^2/s
amplitude = 100.0               # Initial peak temperature excess, K
nx = 51                         # Grid points, including the two boundaries
t_final = 5.0                   # Final time, s
target_r = 0.4                  # Safely below the explicit stability limit

x = np.linspace(0.0, L, nx)
dx = x[1] - x[0]
nsteps = int(np.ceil(t_final / (target_r * dx**2 / alpha)))
dt = t_final / nsteps           # Adjust slightly to land exactly at t_final
r = alpha * dt / dx**2
assert 0.0 < r <= 0.5

save_steps = set(np.linspace(0, nsteps, 5, dtype=int)[1:])

print(f"dx = {dx:.4f} m, dt = {dt:.5f} s, r = {r:.3f}")
print(f"Time steps: {nsteps}")


## Case 1: fixed temperatures at both ends

We first hold both ends at the reference temperature:

$$
\theta(0,t)=\theta(L,t)=0.
$$

The initially warm sinusoidal profile

$$
\theta(x,0)=A\sin(\pi x/L)
$$

already satisfies both boundary conditions.
Physically, the rod is in contact at both ends with ideal temperature reservoirs that remain unaffected by the heat they receive.
Numerically, we update every interior point with the shared explicit formula and then reset both boundary values to zero.

Before revealing or running the output, predict how the peak height and the shape of the temperature profile will change.

In [ ]:
theta_dirichlet = amplitude * np.sin(np.pi * x / L)
theta_dirichlet[[0, -1]] = 0.0
snapshots_dirichlet = [(0.0, theta_dirichlet.copy())]

for step in range(1, nsteps + 1):
    old = theta_dirichlet.copy()
    theta_dirichlet[1:-1] = old[1:-1] + r * (
        old[2:] - 2.0 * old[1:-1] + old[:-2]
    )
    theta_dirichlet[[0, -1]] = 0.0
    if step in save_steps:
        snapshots_dirichlet.append((step * dt, theta_dirichlet.copy()))

print(
    "Final peak temperature excess: "
    f"{theta_dirichlet.max():.2f} K"
)


### Analytical check for Case 1

For this particular initial profile and these boundary conditions, the exact solution is

$$
\theta(x,t)=A\sin(\pi x/L)\exp\left[-\alpha(\pi/L)^2t\right].
$$

Its shape stays sinusoidal while its amplitude decays.
We compare this expression with the numerical solution at the final time.
Agreement here provides a useful check, although it is not a proof that every possible input or implementation is correct.

In [ ]:
exact_dirichlet = amplitude * np.sin(np.pi * x / L) * np.exp(
    -alpha * (np.pi / L)**2 * t_final
)
error_dirichlet = np.max(np.abs(theta_dirichlet - exact_dirichlet))
print(
    f"Maximum absolute error at t = {t_final:g} s: "
    f"{error_dirichlet:.4f} K"
)

fig, ax = plt.subplots()
colors = plt.cm.Blues(np.linspace(0.35, 0.95, len(snapshots_dirichlet)))
for (time, profile), color in zip(snapshots_dirichlet, colors):
    ax.plot(x, profile, color=color, linewidth=1.5,
            zorder=3, label=f"t = {time:.2f} s")
# Plot the exact solution last for its legend position but below the curves.
ax.plot(x, exact_dirichlet, color="black", linestyle=":", linewidth=4,
        zorder=2, label="Exact solution at final time")
ax.set_xlabel("Position / m")
ax.set_ylabel("Temperature excess / K")
ax.set_title("Cooling of a rod with fixed-temperature ends")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()


## Case 2: one fixed end and one insulated end

We now keep the left end at the reference temperature but prevent heat from crossing the right end:

$$
\theta(0,t)=0, \qquad
\left.\frac{\partial\theta}{\partial x}\right|_{x=L}=0.
$$

Because Fourier's law makes conductive heat flux proportional to $-\partial\theta/\partial x$, the zero slope represents perfect insulation.
The compatible initial condition

$$
\theta(x,0)=A\sin\left(\frac{\pi x}{2L}\right)
$$

is zero at the left end and has zero slope at the right end.

### Enforce the zero-flux condition with a ghost point

Let $N$ denote the rightmost physical grid index, so $x_N=L$.
We introduce the auxiliary point $N+1$ one grid spacing beyond the rod and approximate the boundary slope with a centered difference:

$$
\left.\frac{\partial\theta}{\partial x}\right|_{x=L}
\approx\frac{\theta_{N+1}^n-\theta_{N-1}^n}{2\Delta x}=0.
$$

The zero-flux condition therefore supplies the ghost value

$$
\theta_{N+1}^n=\theta_{N-1}^n.
$$

The ghost point is not part of the physical rod and does not need to be stored in the solution array.
It is simply a convenient way to express the missing neighbor required by the centered second derivative at $i=N$.
Substitution into the explicit update gives

$$
\theta_N^{n+1}
=\theta_N^n+r\left(\theta_{N+1}^n-2\theta_N^n+\theta_{N-1}^n\right)
=\theta_N^n+2r\left(\theta_{N-1}^n-\theta_N^n\right).
$$

Interior points still use the common three-point update, the left boundary is reset to zero, and the right boundary uses this ghost-point update.
Before running the code, predict how insulating one end will change the cooling rate and the location of the maximum temperature.

In [ ]:
theta_mixed = amplitude * np.sin(np.pi * x / (2.0 * L))
theta_mixed[0] = 0.0
snapshots_mixed = [(0.0, theta_mixed.copy())]

for step in range(1, nsteps + 1):
    old = theta_mixed.copy()
    theta_mixed[1:-1] = old[1:-1] + r * (
        old[2:] - 2.0 * old[1:-1] + old[:-2]
    )
    theta_mixed[0] = 0.0
    ghost_right = old[-2]
    theta_mixed[-1] = old[-1] + r * (
        ghost_right - 2.0 * old[-1] + old[-2]
    )
    if step in save_steps:
        snapshots_mixed.append((step * dt, theta_mixed.copy()))

print(
    "Final peak temperature excess: "
    f"{theta_mixed.max():.2f} K"
)


### Analytical check for Case 2

The selected half-sine is an eigenfunction of the second-spatial-derivative operator with these mixed boundary conditions.
Its analytical evolution is

$$
\theta(x,t)=A\sin\left(\frac{\pi x}{2L}\right)
\exp\left[-\alpha\left(\frac{\pi}{2L}\right)^2t\right].
$$

The shape remains a half-sine, its maximum stays at the insulated end, and its amplitude decays more slowly than in Case 1 because heat can leave through only one end.

In [ ]:
exact_mixed = amplitude * np.sin(np.pi * x / (2.0 * L)) * np.exp(
    -alpha * (np.pi / (2.0 * L))**2 * t_final
)
error_mixed = np.max(np.abs(theta_mixed - exact_mixed))
print(
    f"Maximum absolute error at t = {t_final:g} s: "
    f"{error_mixed:.4f} K"
)

fig, ax = plt.subplots()
colors = plt.cm.Blues(np.linspace(0.35, 0.95, len(snapshots_mixed)))
for (time, profile), color in zip(snapshots_mixed, colors):
    ax.plot(x, profile, color=color, linewidth=1.5,
            zorder=3, label=f"t = {time:.2f} s")
ax.plot(x, exact_mixed, color="black", linestyle=":", linewidth=4,
        zorder=2, label="Exact solution at final time")
ax.set_xlabel("Position / m")
ax.set_ylabel("Temperature excess / K")
ax.set_title("Cooling with a fixed left end and an insulated right end")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()


## Try next

1. Increase the thermal diffusivity and predict how much faster each case cools before running the notebook again.
2. Refine the grid by increasing `nx`, keeping `target_r` fixed, and compare the final errors and number of time steps.
3. Compare the final temperatures in the two cases and explain the difference using their allowed paths for heat loss.
4. Replace either initial profile with another smooth shape that satisfies its boundary conditions.
   The displayed analytical solutions apply only to the original eigenfunctions, so a general profile would require an eigenfunction series or a different reference solution.
5. Replace the zero-flux condition with a prescribed nonzero flux and derive the corresponding ghost value.

Future extensions could introduce convective boundary conditions, internal heat generation, implicit time stepping, and a systematic convergence study.
